# Version 3 – Retrained Classification Model (Jan+Feb 2021 combined)

**Task:** Retrain the classification model on the combined January + February 2021 dataset.  
**Model:** Random Forest Classifier, trained on combined data. Saved as `models/model_pkl_v3` (Google Drive via DVC).  
**Data:** `data/green_tripdata_2021-01.parquet` + `data/green_tripdata_2021-02.parquet`, `random_state=113`.  
**Version:** Model updated; data unchanged from V2; training code unchanged.

---

***Classification model***
**For classification model, I will predict whether the tip will be given**
I will use the following features:
- passenger count
- trip distance
- pick up hour
- pick up day of the week
- trip fare
Since dataset is balance, based on the tip applied column, no further preprocessing is needed.

I will use the random forest classifier;

**Metrics**
- F1-score
- Accuracy
- Precision
- Recall
- Overall confusion matrix

In [5]:
import sys
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import pickle

sys.path.insert(0, str(Path.cwd().parent))

from HW1.data_preprocessing import load_and_process, train_val_test_split

In [6]:
repo_root = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
data = pd.DataFrame()

january = load_and_process("data/green_tripdata_2021-01.parquet", from_dvc=True, repo=repo_root)
february = load_and_process("data/green_tripdata_2021-02.parquet", from_dvc=True, repo=repo_root)

data = pd.concat([january, february])
data.head()

,passenger_count,trip_distance,fare_amount,pickup_hour,pickup_day_of_week,tip_applied
0,1.0,1.01,5.5,0,4,0
1,1.0,2.53,10.0,0,4,1
2,1.0,1.12,6.0,0,4,1
3,1.0,1.99,8.0,23,3,0
7,6.0,0.45,3.5,0,4,1


In [7]:
train, val, test = train_val_test_split(data, random_state = 113)

In [8]:
print(f"train size: {train.shape}, val size: {val.shape}, test size: {test.shape}")

train size: (49882, 6), val size: (10689, 6), test size: (10690, 6)


In [9]:
# Feature columns (per specification) and target
FEATURE_COLUMNS = [
    "passenger_count",
    "trip_distance",
    "pickup_hour",
    "pickup_day_of_week",
    "fare_amount",
]
TARGET_COLUMN = "tip_applied"

X_train = train[FEATURE_COLUMNS]
X_val = val[FEATURE_COLUMNS]
X_test = test[FEATURE_COLUMNS]

y_train = train[TARGET_COLUMN]
y_val = val[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

In [10]:
X_train.head()

,passenger_count,trip_distance,pickup_hour,pickup_day_of_week,fare_amount
33313,1.0,1.04,8,5,6.0
4966,2.0,5.20,15,1,17.5
16413,1.0,3.53,17,2,16.0
29990,2.0,4.42,19,5,15.0
3710,1.0,17.56,17,0,70.0


In [11]:
param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 6, 10, 20],
    "random_state": [42],
}

In [12]:
results = []
for params in ParameterGrid(param_grid):
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    results.append({
        **params,
        "train_f1": f1_score(y_train, y_train_pred),
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "val_f1": f1_score(y_val, y_val_pred),
        "val_accuracy": accuracy_score(y_val, y_val_pred),
    })

# Store results and best model (by val F1)
best_idx = max(range(len(results)), key=lambda i: results[i]["val_f1"])
best_params = {k: v for k, v in results[best_idx].items() if k in param_grid}
best_f1_score = results[best_idx]["val_f1"]
best_model = RandomForestClassifier(**best_params).fit(X_train, y_train)

In [13]:
results_df = pd.DataFrame(results).sort_values("val_f1", ascending=False)
results_df[["n_estimators", "max_depth", "val_f1", "val_accuracy", "train_f1", "train_accuracy"]]

,n_estimators,max_depth,val_f1,val_accuracy,train_f1,train_accuracy
3,300,3,0.620431,0.555057,0.633365,0.567259
2,200,3,0.618903,0.554121,0.632380,0.566958
1,100,3,0.612890,0.551034,0.630689,0.568261
0,50,3,0.611746,0.551595,0.629331,0.568502
4,50,6,0.592256,0.555711,0.614794,0.577804
5,100,6,0.588286,0.554121,0.614523,0.579207
7,300,6,0.587889,0.554963,0.613076,0.579768
6,200,6,0.586530,0.554308,0.612670,0.579808
11,300,10,0.586482,0.568996,0.635338,0.614510
8,50,10,0.586407,0.566751,0.634942,0.612906


In [14]:
print("Best parameters:", best_params)
print("Best validation F1:", best_f1_score)
y_val_pred = best_model.predict(X_val)
print("Validation F1:", f1_score(y_val, y_val_pred))
print("Validation accuracy:", accuracy_score(y_val, y_val_pred))

Best parameters: {'max_depth': 3, 'n_estimators': 300, 'random_state': 42}
Best validation F1: 0.6204309656823623
Validation F1: 0.6204309656823623
Validation accuracy: 0.5550566002432407


In [15]:
# Evaluate best model on test set
y_test_pred = best_model.predict(X_test)

print("Test set metrics:")
print("F1-score:", f1_score(y_test, y_test_pred))
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test set metrics:
F1-score: 0.6277711084433774
Accuracy: 0.5649204864359214
Precision: 0.5349154391707583
Recall: 0.7596358706178579

Confusion matrix:
[[2117 3410]
 [1241 3922]]


In [16]:
with open('models/model_pkl_v3', 'wb') as files:
    pickle.dump(best_model, files)